# P04 · Нейросети и обучение / Neural networks and training


**Данные / Data:** [Digits](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html), реальные рукописные цифры из коллекции UCI / real handwritten digits from the UCI collection. 1,797 images, 8×8 pixels, intensities 0–16, 10 classes. Это не MNIST / This is not MNIST.

**Объект / Observation:** одно изображение / one image. **Признаки / Features:** 64 яркости, доступные при прогнозе / 64 intensities available at prediction time. **Цель / Target:** цифра 0–9 / digit 0–9.

**Ограничение / Limitation:** load_digits не содержит writer ID: разбиение по изображениям не доказывает перенос на новых авторов. / No writer IDs are supplied: an image split does not establish generalisation to unseen writers.

**Запуск / Runtime:** Google Colab, CPU, «Выполнить всё / Run all». Библиотеки Colab уже установлены / Colab includes the required libraries. Порядок: Train → Validation → зафиксированный выбор / frozen choice → Test.

In [ ]:
import copy, json, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import log_loss, accuracy_score
SEED = 2026
CLASSES = np.arange(10)
print({'python': platform.python_version(), 'numpy': np.__version__,
       'sklearn': sklearn.__version__})
digits = load_digits()
X = digits.data.astype(np.float64) / 16.0  # Fixed format bound / Известная граница формата
y = digits.target
row_ids = np.arange(len(y))
assert X.shape == (1797, 64) and y.shape == (1797,)
assert 0 <= X.min() <= X.max() <= 1
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for label, ax in enumerate(axes.flat):
    i = np.flatnonzero(y == label)[0]
    ax.imshow(digits.images[i], cmap='gray_r', vmin=0, vmax=16)
    ax.set_title(str(label)); ax.axis('off')
plt.tight_layout(); plt.show()

## 1. Разделим изображения / Split the images

60/20/20, stratified; seed=2026. Не создаём искусственные группы / Do not invent artificial groups.

In [ ]:
trainval_ids, test_ids = train_test_split(row_ids, test_size=0.2, stratify=y, random_state=SEED)
train_ids, val_ids = train_test_split(trainval_ids, test_size=0.25,
                                     stratify=y[trainval_ids], random_state=SEED)
assert not (set(train_ids) & set(val_ids) or set(train_ids) & set(test_ids) or set(val_ids) & set(test_ids))
assert set(np.concatenate([train_ids, val_ids, test_ids])) == set(row_ids)
print({'Train': len(train_ids), 'Validation': len(val_ids), 'Test': len(test_ids)})
X_train, y_train = X[train_ids], y[train_ids]
X_val, y_val = X[val_ids], y[val_ids]
# Test remains unused until final evaluation / Test не используется до финальной оценки.

## 2. Baseline

Фиксируем C=1 до сравнения / Fix C=1 before comparison.

In [ ]:
baseline = LogisticRegression(C=1.0, max_iter=2000, random_state=SEED)
baseline.fit(X_train, y_train)
print('Baseline validation:', {'loss': log_loss(y_val, baseline.predict_proba(X_val), labels=CLASSES),
                               'accuracy': baseline.score(X_val, y_val)})

## 3. Обучение по эпохам / Epoch-by-epoch training

partial_fit сохраняет состояние Adam. Сохраняем deepcopy лучшей модели по Validation loss; выполняем все 60 эпох. / partial_fit preserves Adam state. Save a deep copy with minimum Validation loss; run all 60 epochs.

Кривые: одинаковая cross-entropy без штрафа L2. Обучающая цель содержит L2. / Curves use the same cross-entropy without L2; the training objective includes L2.

In [ ]:
def train_run(X_train, y_train, X_val, y_val, width, lr, epochs=60, seed=SEED):
    model = MLPClassifier(hidden_layer_sizes=(width,), activation='relu', solver='adam',
                          alpha=0.0001, batch_size=64, learning_rate_init=lr,
                          shuffle=True, random_state=seed, early_stopping=False)
    history = []
    best_model, best_loss, best_epoch = None, np.inf, None
    for epoch in range(1, epochs + 1):
        # partial_fit: one epoch, retaining optimiser state / Одна эпоха с сохранением состояния Adam.
        model.partial_fit(X_train, y_train, classes=CLASSES)
        p_train = model.predict_proba(X_train)
        p_val = model.predict_proba(X_val)
        tr_loss = log_loss(y_train, p_train, labels=CLASSES)
        va_loss = log_loss(y_val, p_val, labels=CLASSES)
        history.append({'epoch': epoch, 'train_loss': tr_loss, 'val_loss': va_loss,
                        'val_accuracy': accuracy_score(y_val, model.predict(X_val))})
        if va_loss < best_loss:
            best_loss, best_epoch = va_loss, epoch
            best_model = copy.deepcopy(model)  # Independent checkpoint / Независимая копия
    return {'model': best_model, 'history': pd.DataFrame(history), 'width': width,
            'lr': lr, 'best_epoch': best_epoch, 'val_loss': best_loss}

In [ ]:
configs = [(32, 0.001), (32, 0.01)]

runs = [train_run(X_train, y_train, X_val, y_val, width, lr, epochs=60)
        for width, lr in configs]
# Deterministic tie rule: first configuration / При равенстве выбираем первый вариант.
best = min(runs, key=lambda run: run['val_loss'])
summary = pd.DataFrame([{k: run[k] for k in ['width', 'lr', 'best_epoch', 'val_loss']} for run in runs])
display(summary)
fig, axes = plt.subplots(1, len(runs), figsize=(5 * len(runs), 4), squeeze=False)
for ax, run in zip(axes.flat, runs):
    h = run['history']
    ax.plot(h.epoch, h.train_loss, label='Train')
    ax.plot(h.epoch, h.val_loss, label='Validation')
    ax.axvline(run['best_epoch'], color='gray', linestyle='--', label='Selected epoch')
    ax.set(title=f"width={run['width']}, lr={run['lr']}", xlabel='Epoch', ylabel='Cross-entropy')
    ax.legend()
plt.tight_layout(); plt.show()
selection = {k: best[k] for k in ['width', 'lr', 'best_epoch', 'val_loss']}
selection['criterion'] = 'minimum validation cross-entropy; first config on ties'
print('Frozen selection:', selection)

## 4. Проверим прямой проход / Verify the forward pass

Используем реальные изображения Validation и веса обученной сети. / Use real validation images and learned weights.

In [ ]:
model = best['model']
xb = X_val[:7]
hidden = np.maximum(0, xb @ model.coefs_[0] + model.intercepts_[0])
logits = hidden @ model.coefs_[1] + model.intercepts_[1]
exp_logits = np.exp(logits - logits.max(axis=1, keepdims=True))
manual_probabilities = exp_logits / exp_logits.sum(axis=1, keepdims=True)
assert np.allclose(manual_probabilities, model.predict_proba(xb))
parameter_count = sum(w.size for w in model.coefs_) + sum(b.size for b in model.intercepts_)
assert parameter_count == 2410
print('Parameters:', parameter_count, 'output shape:', manual_probabilities.shape)

## 5. Финальная оценка / Final evaluation

Выбор выше уже зафиксирован. Test не используется для изменения параметров. / The choice above is frozen. Do not change settings using Test.

In [ ]:
# Execute only after recording selection / Выполнить только после фиксации выбора.
results = []
for name, model in [('LogisticRegression', baseline), ('MLP', best['model'])]:
    probabilities = model.predict_proba(X[test_ids])
    assert probabilities.shape == (len(test_ids), 10)
    assert np.allclose(probabilities.sum(axis=1), 1)
    results.append({'model': name, 'test_loss': log_loss(y[test_ids], probabilities, labels=CLASSES),
                    'test_accuracy': accuracy_score(y[test_ids], model.predict(X[test_ids]))})
display(pd.DataFrame(results))
# Do not tune on these results / Не подбирайте настройки по этим результатам.

## Обсуждение / Discussion

1. Почему loss и accuracy могут ранжировать модели по-разному? / Why can loss and accuracy rank models differently?
2. Совпала ли выбранная эпоха с последней? / Was the selected epoch the final one?
3. Доказывает ли этот split перенос на новых авторов? / Does this split establish generalisation to unseen writers?
4. Для проекта укажите данные, baseline и критерий выбора. / Specify your project data, baseline and selection criterion.